# Pandas Basics

## Overview

Pandas is the foundation of data wrangling in Python. Two core data structures:

| Structure | Description | R equivalent |
|---|---|---|
| `Series` | 1D labelled array | Named vector |
| `DataFrame` | 2D labelled table | `data.frame` / tibble |

**Core workflow:** load → inspect → select → filter → mutate → summarise → export.

**Two indexing systems:**
- `.loc[]` — label-based; uses row/column names
- `.iloc[]` — position-based; uses integer positions

Always be explicit about which you are using. Mixing them is a common source of silent errors.

---

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.3f}'.format)

rng = np.random.default_rng(42)

# Simulate: 120 riparian monitoring sites
n = 120
sites = pd.DataFrame({
    'site_id':    [f'SITE_{i:03d}' for i in range(1, n+1)],
    'catchment':  rng.choice(['North', 'South', 'East', 'West'], size=n),
    'elevation':  rng.uniform(50, 400, n).round(1),
    'nitrate':    rng.gamma(2, 2, n).round(2),
    'phosphorus': rng.gamma(1.5, 1.5, n).round(2),
    'richness':   rng.integers(5, 35, n),
    'treatment':  rng.choice(['control', 'restored'], size=n),
})

print(f'Shape: {sites.shape}')
sites.head()

Shape: (120, 7)


,site_id,catchment,elevation,nitrate,phosphorus,richness,treatment
0,SITE_001,North,283.900,3.760,3.270,6,restored
1,SITE_002,West,214.900,4.130,2.090,16,restored
2,SITE_003,East,247.800,3.930,4.900,19,control
3,SITE_004,South,317.700,7.080,0.780,32,control
4,SITE_005,South,272.200,9.080,1.600,11,restored


---
## Inspecting a DataFrame

In [2]:
sites.info()                                          # dtypes, non-null counts
print(sites.describe())                               # numeric summary
print(sites['catchment'].value_counts())              # categorical counts
print(sites['treatment'].value_counts(normalize=True).round(3))  # proportions
print('\nMissing values:')
print(sites.isnull().sum())

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   site_id     120 non-null    str    
 1   catchment   120 non-null    str    
 2   elevation   120 non-null    float64
 3   nitrate     120 non-null    float64
 4   phosphorus  120 non-null    float64
 5   richness    120 non-null    int64  
 6   treatment   120 non-null    str    
dtypes: float64(3), int64(1), str(3)
memory usage: 6.7 KB
       elevation  nitrate  phosphorus  richness
count    120.000  120.000     120.000   120.000
mean     218.030    4.526       2.132    18.958
std       97.841    3.195       1.726     8.958
min       57.600    0.210       0.030     5.000
25%      142.225    2.522       0.925    12.000
50%      217.250    3.945       1.625    18.000
75%      296.400    5.923       2.953    27.250
max      397.300   17.870       9.330    34.000
catchment
West     34
East     29
South    29
No

---
## Selecting Rows and Columns

In [3]:
# Column selection
sites[['site_id', 'richness', 'treatment']].head(3)

# Select columns by dtype
numeric_cols = sites.select_dtypes(include='number').columns.tolist()
print('Numeric columns:', numeric_cols)

# Boolean filtering with .loc
high_nitrate = sites.loc[sites['nitrate'] > 5]
print(f'High nitrate sites: {len(high_nitrate)}')

# Multiple conditions — each wrapped in parentheses
restored_north = sites.loc[
    (sites['treatment'] == 'restored') & (sites['catchment'] == 'North')
]
print(f'Restored North sites: {len(restored_north)}')

# .iloc for positional access
print('\nFirst 3 rows, first 4 columns:')
print(sites.iloc[:3, :4])

# query() for readable filtering
rich_restored = sites.query('richness > 25 and treatment == "restored"')
print(f'Rich restored sites: {len(rich_restored)}')

Numeric columns: ['elevation', 'nitrate', 'phosphorus', 'richness']
High nitrate sites: 39
Restored North sites: 12

First 3 rows, first 4 columns:
    site_id catchment  elevation  nitrate
0  SITE_001     North    283.900    3.760
1  SITE_002      West    214.900    4.130
2  SITE_003      East    247.800    3.930
Rich restored sites: 17


---
## Creating and Transforming Columns

In [4]:
# assign(): method-chaining friendly — equivalent to dplyr::mutate
sites = (
    sites
    .assign(
        log_nitrate    = lambda df: np.log1p(df['nitrate']),
        nutrient_ratio = lambda df: df['nitrate'] / (df['phosphorus'] + 0.01),
        elev_class     = lambda df: pd.cut(
            df['elevation'],
            bins=[0, 150, 300, np.inf],
            labels=['low', 'mid', 'high']
        ),
        richness_std   = lambda df: (
            (df['richness'] - df['richness'].mean()) / df['richness'].std()
        )
    )
)
print(sites[['nitrate', 'log_nitrate', 'nutrient_ratio', 'elev_class']].head(5))

# np.where() for conditional column — faster than apply() for binary logic
sites['richness_label'] = np.where(
    sites['richness'] > 25, 'high',
    np.where(sites['richness'] < 12, 'low', 'medium')
)
print(sites['richness_label'].value_counts())

   nitrate  log_nitrate  nutrient_ratio elev_class
0    3.760        1.560           1.146        mid
1    4.130        1.635           1.967        mid
2    3.930        1.595           0.800        mid
3    7.080        2.089           8.962       high
4    9.080        2.311           5.640        mid
richness_label
medium    60
high      33
low       27
Name: count, dtype: int64


---
## Grouping and Summarising

In [5]:
# groupby + agg: named aggregation
summary = (
    sites
    .groupby(['catchment', 'treatment'])
    .agg(
        n             = ('site_id',   'count'),
        richness_mean = ('richness',  'mean'),
        richness_sd   = ('richness',  'std'),
        nitrate_median= ('nitrate',   'median'),
    )
    .round(2)
    .reset_index()   # promotes grouping columns back to regular columns
)
print(summary)

# transform(): adds group summary back to original row count
# Equivalent to dplyr::mutate after group_by
sites['catchment_mean_richness'] = (
    sites.groupby('catchment')['richness'].transform('mean')
)
sites['richness_within_group'] = (
    sites['richness'] - sites['catchment_mean_richness']
)
print(sites[['site_id','catchment','richness','catchment_mean_richness','richness_within_group']].head(5))

  catchment treatment   n  richness_mean  richness_sd  nitrate_median
0      East   control  20         16.450        8.850           4.000
1      East  restored   9         23.670        9.910           3.480
2     North   control  16         15.060        7.800           5.350
3     North  restored  12         19.750        9.970           3.100
4     South   control  14         19.430        8.610           4.690
5     South  restored  15         21.470        8.250           4.760
6      West   control  21         19.380        8.780           3.570
7      West  restored  13         19.540        9.690           3.090
    site_id catchment  richness  catchment_mean_richness  \
0  SITE_001     North         6                   17.071   
1  SITE_002      West        16                   19.441   
2  SITE_003      East        19                   18.690   
3  SITE_004     South        32                   20.483   
4  SITE_005     South        11                   20.483   

   richne

---
## Sorting and Ranking

In [6]:
sites_sorted = sites.sort_values(
    ['catchment', 'richness'], ascending=[True, False]
)

# Rank within groups
sites['richness_rank'] = (
    sites.groupby('catchment')['richness']
    .rank(method='dense', ascending=False)
    .astype(int)
)

print(f'Final shape: {sites.shape}')
print(f'Columns: {sites.columns.tolist()}')

Final shape: (120, 15)
Columns: ['site_id', 'catchment', 'elevation', 'nitrate', 'phosphorus', 'richness', 'treatment', 'log_nitrate', 'nutrient_ratio', 'elev_class', 'richness_std', 'richness_label', 'catchment_mean_richness', 'richness_within_group', 'richness_rank']


---

## Common Pitfalls

**1. Chained indexing (`df['col'][mask]`) instead of `.loc`**  
Chained indexing may operate on a copy rather than the original DataFrame — assignments silently fail and pandas raises a `SettingWithCopyWarning`. Always use `df.loc[condition, 'col']` for combined row filtering and column assignment.

**2. Modifying a DataFrame during iteration**  
Iterating with `iterrows()` and modifying the DataFrame in the loop is slow and error-prone. Almost all row-wise operations have a vectorised pandas or numpy equivalent. If custom logic is unavoidable, use `assign()` or `apply()` rather than modifying in place during iteration.

**3. Forgetting `reset_index()` after `groupby().agg()`**  
After aggregation, grouping columns become the index. Accessing them as regular columns then raises a `KeyError`. Always call `.reset_index()` after aggregation unless you specifically need a MultiIndex.

**4. Using `apply()` where vectorised operations exist**  
`df['col'].apply(lambda x: x * 2)` is orders of magnitude slower than `df['col'] * 2`. `apply()` is a Python loop under the hood. Reserve it for genuinely complex per-row logic that cannot be expressed vectorially; prefer `np.where()` for conditionals and built-in pandas methods for everything else.

**5. Not checking dtypes after loading**  
Pandas infers dtypes on read. Numeric columns stored as strings, integers read as floats due to missing values, and categories read as `object` are all common. Always run `.info()` immediately after loading and cast explicitly with `.astype()` before proceeding.

---
*python_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*